In [1]:
import regex as re
import pandas as pd

In [85]:
def solo_emojis(texto: str) -> bool:
    """
    Retorna True si el texto contiene solo emojis (y caracteres invisibles como ZWJ o VS16),
    y False si hay algún otro carácter (letras, números, espacios, puntuación, etc.).
    """
    # Patrón que coincide con cualquier emoji (incluyendo secuencias complejas)
    patron_emoji = re.compile(r'\p{Emoji_Presentation}|\p{Extended_Pictographic}')
    
    # Eliminar posibles caracteres de control/unión que no son emojis por sí mismos
    texto_limpio = re.sub(r'[\u200d\uFE0F\uFE0E\u061C\u200E\u200F\u202A-\u202E]', '', texto)
    
    if not texto_limpio:
        return False
    
    # Buscar todos los "trozos" que no son emojis (caracteres normales)
    resto = re.sub(patron_emoji, '', texto_limpio)
    
    # Si después de quitar emojis queda algo, hay caracteres no-emoji
    return len(resto.strip()) == 0



def remove_special_characters(text: str) -> str:
    """Elimina caracteres de control, basura Unicode, emojis y etiquetas [STICKER], preservando acentos y ñ."""
    
    if not isinstance(text, str):
        return text
    
    # 1. Eliminar etiquetas [STICKER] (NUEVO)
    # Esto elimina [STICKER] exacto, insensible a mayúsculas/minúsculas
    #text = re.sub(r'\[STICKER\]', '', text, flags=re.IGNORECASE)
    # También puedes eliminar variantes como [Sticker], [sticker], etc. (ya cubierto con IGNORECASE)
    
    # También eliminar si hay espacios alrededor (opcional)
    # text = re.sub(r'\s*\[STICKER\]\s*', ' ', text, flags=re.IGNORECASE)
    
    # 2. Eliminar caracteres de control (excepto newlines y tabs)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    
    # 3. Reemplazar caracteres Unicode problemáticos comunes
    replacements = {
        "\u2018": "'", "\u2019": "'",  # Comillas simples tipográficas
        "\u201c": '"', "\u201d": '"',  # Comillas dobles tipográficas
        "\u2013": "-", "\u2014": "-",  # Guiones em/en
        "\u2026": "...",               # Elipsis
        "\u00a0": " ",                 # Non-breaking space
        "\ufeff": "",                  # BOM
        "\u200b": "",                  # Zero-width space
        "\uf0b7": "- ",                # Bullet point (symbol font)
        "\uf0a7": "- ",                # Otro bullet
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    
    # 4. Eliminar emojis
    # Esta regex captura la mayoría de emojis, incluyendo:
    # - Emojis básicos (😀, ❤️)
    # - Emojis con modificadores de tono de piel (👋🏽)
    # - Emojis ZWJ (familia, profesiones: 👨‍👩‍👧‍👦, 👩‍💻)
    # - Símbolos de flechas y otros (⚠️, ㊗️)
    # - Números y letras rodeados (▶️, ℹ️)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # Emojis emociones
        "\U0001F300-\U0001F5FF"  # Símbolos y pictogramas
        "\U0001F680-\U0001F6FF"  # Transporte y mapas
        "\U0001F700-\U0001F77F"  # Símbolos alquímicos
        "\U0001F780-\U0001F7FF"  # Símbolos geométricos extendidos
        "\U0001F800-\U0001F8FF"  # Flechas suplementarias-C
        "\U0001F900-\U0001F9FF"  # Emojis suplementarios (2020+)
        "\U0001FA00-\U0001FA6F"  # Ajedrez y símbolos extendidos
        "\U0001FA70-\U0001FAFF"  # Emojis adicionales (2021+)
        "\U00002702-\U000027B0"  # Símbolos dingbat
        "\U000024C2-\U0001F251"  # Símbolos encerrados
        "]+",
        flags=re.UNICODE
    )
    
    # También capturar emojis con modificadores ZWJ (familias, etc.)
    # Eliminamos primero los que tienen joiners y variantes
    text = re.sub(r'[\U0001F3FB-\U0001F3FF]', '', text)  # Tono de piel
    text = re.sub(r'\u200D', '', text)  # Zero-width joiner
    text = emoji_pattern.sub(r'', text)
    
    # 5. Limpieza final: eliminar espacios múltiples que puedan quedar
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text



def indices(datos: str) -> str:
    #limpiamos celdas vacias
    borrar = []
    for i in range(len(datos)):
        if ((type(datos[i]) != str)):
            borrar.append(i)
        elif solo_emojis(datos[i]):
            borrar.append(i)
        elif (datos[i] == '[Sticker]' or datos[i] == '[Sticker] '):
            borrar.append(i)
        elif (datos[i] == ''):
            borrar.append(i)
    return borrar

In [86]:
#modificar path en caso de correrse de forma local en otra maquina
path_raw = 'C:/Users/inqui/OneDrive/Desktop/Clases/26-2/LLM_PROJECT_1/data/raw/'
path_proc = 'C:/Users/inqui/OneDrive/Desktop/Clases/26-2/LLM_PROJECT_1/data/processed/'

archivo = ['tiktok infraestructura_evaluacion_humano','tiktok turismo_evaluacion_humano']#,'tiktok seguridad_evaluacion_humano']
archivo_fin = ['infraestructura','turismo']#,'seguridad']
for i in range(len(archivo)):
    aux = path_raw + archivo[i] + '.csv'
    #print(aux)
    datos = pd.read_csv(aux, encoding = 'latin-1')
    #print(datos.head())
    
    datos_limpios = datos.drop(indices(datos['comentario']))
    datos_limpios['comentario'] = datos_limpios['comentario'].astype(str).apply(remove_special_characters)
    
    aux2 = path_proc + archivo_fin[i] + '_limpio.csv'
    #print(aux2)
    datos_limpios.to_csv(aux2, encoding = 'utf-8')
print('Limpieza de comentarios terminada')

Limpieza de comentarios terminada
